# NOTEBOOK 02 – NGỌC LAN | CHƯƠNG 2: CƠ SỞ LÝ THUYẾT

**Phụ trách:** giữ nguyên 2.1–2.5, đồng thời thực hành huấn luyện **Random Forest** và đo **Precision, Recall, F1, AP/PR-AUC, FP, Top-K trên Validation**. Issue **#3**. Lý thuyết 12/11, RF bàn giao trước 20/11/2026.

**Mục đích:** tạo phần lý thuyết chuẩn, có thể giải thích khi bảo vệ; phụ trách mô hình Random Forest; **không dùng Test để chọn mô hình/ngưỡng**. Trúc Chi phụ trách Dummy Classifier và tích hợp mô hình; Tấn Lên huấn luyện MLP; Ngọc Lan huấn luyện Random Forest; Vĩnh Chi đánh giá độc lập cả ba mô hình trên Test.

[Tệp Markdown lý thuyết chi tiết](../docs/theory/02_ngoc_lan_ch2.md) · [Nguồn](../docs/REFERENCES.md).

## 2.1 Cơ sở về TMĐT và logistics
Mô tả vòng đời đơn đặt hàng → phê duyệt thanh toán → chuẩn bị hàng → giao cho vận chuyển → giao đến khách.

**Khái niệm:** `estimated delivery date` = ngày giao dự kiến; `actual delivery date` = ngày giao thực tế; `SLA` = thời hạn cam kết theo điều kiện dịch vụ.

**Tại sao dự báo sớm:** bộ phận vận hành cần ưu tiên theo dõi khi còn khả năng can thiệp. Việc phân biệt **dự báo trước** và **giải thích sau** là bắt buộc.

## 2.2 Khai thác dữ liệu và EDA
1. Kiểm tra schema, độ đầy đủ, khóa chính, khóa ngoại, quan hệ 1–n.
2. Chọn đơn vị phân tích **order_id** và tạo nhãn nhị phân bằng ngày lịch.
3. Thống kê đơn, đơn trễ, phân phối giá/phí vận chuyển, hành vi theo thời gian, bang và nhóm hàng.
4. Trực quan hóa có diễn giải: biểu đồ, bảng tỷ lệ, số mẫu của từng nhóm.
5. Phân biệt tương quan trong EDA và khả năng dự báo tại `order_approved_at`.

Cần liên kết hình và kết quả EDA do Lam Linh thực nghiệm, không tự đặt số liệu.

### 2.2.1. Thực hành T05 – Kiểm tra chất lượng dữ liệu thô

**Người thực hiện:** Ngọc Lan  
**Thành viên phối hợp:** Lam Linh  
**Nguồn dữ liệu:** Brazilian E-Commerce Public Dataset by Olist.

#### Mục tiêu

Kiểm tra chất lượng của 9 tệp CSV trước khi dữ liệu
được làm sạch, tích hợp và sử dụng để xây dựng mô hình
dự đoán nguy cơ giao hàng trễ.

Nội dung kiểm tra bao gồm:

1. Số dòng và số cột của từng bảng.
2. Số lượng và tỷ lệ giá trị thiếu (Missing Values).
3. Số lượng bản ghi trùng hoàn toàn (Duplicate Rows).
4. Kiểu dữ liệu của từng thuộc tính (Data Types).
5. Một số giá trị bất thường và tính hợp lệ của khóa.


#### Phương pháp

Sử dụng thư viện Pandas với các phương thức và thuộc tính:
`read_csv()`, `isna()`, `duplicated()`, `dtypes`,
`nunique()`, `describe()` và `value_counts()`.

Kết quả được thống kê theo từng bảng và từng cột,
làm cơ sở đánh giá chất lượng dữ liệu thô.

In [1]:
# T05.1 - Thiết lập môi trường

from pathlib import Path
import pandas as pd
from IPython.display import display

# Tìm thư mục gốc repository
HERE = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [HERE, *HERE.parents]
        if (p / "data" / "raw").is_dir()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw trong repository."
    )

RAW_DIR = ROOT / "data" / "raw"

FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

missing_files = [
    filename for filename in FILES.values()
    if not (RAW_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Thiếu dữ liệu: {missing_files}"
    )

print("Thư mục dự án:", ROOT)
print("Số file tìm thấy:", len(FILES))
print("Kết quả: Đã tìm thấy đủ 9 file CSV.")


Thư mục dự án: D:\UTH HK1 NAM 3\ĐO AN\Do_An_Khoa_hoc_du_lieu---GR02
Số file tìm thấy: 9
Kết quả: Đã tìm thấy đủ 9 file CSV.


In [2]:
# T05.2 - Data Profiling toàn bộ 9 bảng

qa_summary = []
qa_columns = []

for table_name, filename in FILES.items():
    file_path = RAW_DIR / filename

    # Giữ các mã định danh và ZIP ở dạng string
    cols = pd.read_csv(file_path, nrows=0).columns

    dtype_map = {
        col: "string" for col in cols
        if (
            (col.endswith("_id")
             and col != "order_item_id")
            or "zip_code_prefix" in col
        )
    }

    data = pd.read_csv(
        file_path,
        dtype=dtype_map,
        low_memory=False
    )

    total_missing = int(data.isna().sum().sum())
    total_cells = data.size

    qa_summary.append({
        "table": table_name,
        "n_rows": len(data),
        "n_columns": len(data.columns),
        "missing_cells": total_missing,
        "missing_pct": round(
            100 * total_missing / total_cells, 3
        ),
        "duplicate_rows": int(
            data.duplicated().sum()
        ),
        "columns_with_missing": int(
            (data.isna().sum() > 0).sum()
        )
    })

    # Kiểm tra từng cột
    for col in data.columns:
        qa_columns.append({
            "table": table_name,
            "column": col,
            "dtype": str(data[col].dtype),
            "missing": int(data[col].isna().sum()),
            "missing_pct": round(
                100 * data[col].isna().mean(), 3
            ),
            "n_unique": int(
                data[col].nunique(dropna=True)
            )
        })

    print(f"Đã kiểm tra: {table_name}")

# Tổng hợp
qa_summary_df = pd.DataFrame(qa_summary)
qa_columns_df = pd.DataFrame(qa_columns)

assert len(qa_summary_df) == 9

print("\nBẢNG TỔNG HỢP QA - 9 CSV")
display(qa_summary_df)

print("\nTỔNG SỐ CỘT ĐÃ KIỂM TRA:",
      len(qa_columns_df))


Đã kiểm tra: orders
Đã kiểm tra: customers
Đã kiểm tra: items
Đã kiểm tra: payments
Đã kiểm tra: reviews
Đã kiểm tra: products
Đã kiểm tra: sellers
Đã kiểm tra: geolocation
Đã kiểm tra: translation

BẢNG TỔNG HỢP QA - 9 CSV


,table,n_rows,n_columns,missing_cells,missing_pct,duplicate_rows,columns_with_missing
0,orders,99441,8,4908,0.617,0,3
1,customers,99441,5,0,0.000,0,0
2,items,112650,7,0,0.000,0,0
3,payments,103886,5,0,0.000,0,0
4,reviews,99224,7,145903,21.006,0,2
5,products,32951,9,2448,0.825,0,8
6,sellers,3095,4,0,0.000,0,0
7,geolocation,1000163,5,0,0.000,261831,0
8,translation,71,2,0,0.000,0,0



TỔNG SỐ CỘT ĐÃ KIỂM TRA: 52


In [3]:
# T05.3 - Phân tích chi tiết theo từng cột

missing_detail = (
    qa_columns_df[
        qa_columns_df["missing"] > 0
    ]
    .sort_values(
        ["table", "missing"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("CÁC CỘT CÓ GIÁ TRỊ THIẾU")
display(missing_detail)

print("\nKIỂU DỮ LIỆU CỦA TẤT CẢ CÁC CỘT")
display(
    qa_columns_df[
        ["table", "column", "dtype", "n_unique"]
    ]
)

print("\nSỐ CỘT THEO KIỂU DỮ LIỆU")
display(
    qa_columns_df.groupby("dtype")
    .size()
    .reset_index(name="n_columns")
)


CÁC CỘT CÓ GIÁ TRỊ THIẾU


,table,column,dtype,missing,missing_pct,n_unique
0,orders,order_delivered_customer_date,object,2965,2.982,95664
1,orders,order_delivered_carrier_date,object,1783,1.793,81018
2,orders,order_approved_at,object,160,0.161,90733
3,products,product_category_name,object,610,1.851,73
4,products,product_name_lenght,float64,610,1.851,66
5,products,product_description_lenght,float64,610,1.851,2960
6,products,product_photos_qty,float64,610,1.851,19
7,products,product_weight_g,float64,2,0.006,2204
8,products,product_length_cm,float64,2,0.006,99
9,products,product_height_cm,float64,2,0.006,102



KIỂU DỮ LIỆU CỦA TẤT CẢ CÁC CỘT


,table,column,dtype,n_unique
0,orders,order_id,string,99441
1,orders,customer_id,string,99441
2,orders,order_status,object,8
3,orders,order_purchase_timestamp,object,98875
4,orders,order_approved_at,object,90733
5,orders,order_delivered_carrier_date,object,81018
6,orders,order_delivered_customer_date,object,95664
7,orders,order_estimated_delivery_date,object,459
8,customers,customer_id,string,99441
9,customers,customer_unique_id,string,96096



SỐ CỘT THEO KIỂU DỮ LIỆU


,dtype,n_columns
0,float64,12
1,int64,4
2,object,21
3,string,15


### 2.2.2. Nhận xét kết quả Data Profiling

Quá trình kiểm tra 9 bảng Olist ghi nhận:

- **Orders:** Có 4.908 ô thiếu, tập trung ở các trường
  thời gian phê duyệt, bàn giao vận chuyển và giao hàng.
  Các trường này cần được kiểm tra khi chọn mẫu và tạo nhãn.
- **Reviews:** Có 145.903 ô thiếu, chủ yếu ở tiêu đề
  và nội dung bình luận. Đây là các thông tin không bắt
  buộc, vì vậy không nên xóa toàn bộ bản ghi.
- **Products:** Có 2.448 ô thiếu liên quan đến danh mục,
  mô tả và thông số sản phẩm.
- **Geolocation:** Có 261.831 dòng trùng hoàn toàn.
  Cần xử lý phù hợp trước khi tổng hợp theo mã ZIP.

Các bảng còn lại không phát hiện giá trị thiếu hoặc
dòng trùng hoàn toàn trong bước profiling.

**Nhận xét:** Dữ liệu cần được xử lý theo ý nghĩa
nghiệp vụ của từng cột. Không xem mọi giá trị thiếu
hoặc khóa `order_id` lặp trong bảng chi tiết là lỗi.

**Đề xuất xử lý:**

- **Orders:** Kiểm tra thời gian thiếu trước khi chọn mẫu
  và xác định nhãn giao hàng trễ.
- **Reviews:** Không tự động loại bỏ bản ghi chỉ vì thiếu
  nội dung bình luận; không sử dụng đánh giá sau giao
  hàng làm đặc trưng dự báo.
- **Products:** Đánh giá giá trị thiếu theo từng thuộc tính
  trước khi quyết định điền thiếu hoặc loại bỏ.
- **Geolocation:** Kiểm tra trùng lặp và tổng hợp dữ liệu
  phù hợp theo ZIP prefix trước khi kết nối bảng.

Các đề xuất này được bàn giao cho Lam Linh để đối chiếu
với quy trình làm sạch trong Chương 3. Không thay đổi
dữ liệu gốc tại bước T05.

In [4]:
# T05.4 - Kiểm tra khóa và giá trị bất thường

qa_checks = []

def add_check(table, check_name, affected, note):
    qa_checks.append({
        "table": table,
        "check": check_name,
        "affected_rows": int(affected),
        "note": note
    })

# 1. Kiểm tra order_id ở bảng orders
orders = pd.read_csv(
    RAW_DIR / FILES["orders"],
    dtype={"order_id": "string"}
)

add_check(
    "orders", "missing_order_id",
    orders["order_id"].isna().sum(),
    "Khóa đơn hàng không được thiếu"
)

add_check(
    "orders", "duplicate_order_id",
    orders["order_id"].duplicated().sum(),
    "order_id phải duy nhất trong bảng orders"
)

# 2. Kiểm tra bảng items
items = pd.read_csv(
    RAW_DIR / FILES["items"]
)

add_check(
    "items", "negative_price",
    (items["price"] < 0).sum(),
    "Giá sản phẩm âm là bất thường"
)

add_check(
    "items", "negative_freight",
    (items["freight_value"] < 0).sum(),
    "Phí vận chuyển âm là bất thường"
)

add_check(
    "items", "duplicate_item_key",
    items.duplicated(
        ["order_id", "order_item_id"]
    ).sum(),
    "Cặp order_id và order_item_id phải duy nhất"
)

# 3. Kiểm tra điểm đánh giá
reviews = pd.read_csv(
    RAW_DIR / FILES["reviews"]
)

invalid_review = (
    reviews["review_score"].notna()
    & ~reviews["review_score"].isin([1, 2, 3, 4, 5])
)

add_check(
    "reviews", "invalid_review_score",
    invalid_review.sum(),
    "Điểm đánh giá hợp lệ từ 1 đến 5"
)

# 4. Kiểm tra thanh toán
payments = pd.read_csv(
    RAW_DIR / FILES["payments"]
)

add_check(
    "payments", "negative_payment",
    (payments["payment_value"] < 0).sum(),
    "Giá trị thanh toán âm là bất thường"
)

add_check(
    "payments", "duplicate_payment_key",
    payments.duplicated(
        ["order_id", "payment_sequential"]
    ).sum(),
    "Khóa ghép thanh toán phải duy nhất"
)

# Tổng hợp
qa_checks_df = pd.DataFrame(qa_checks)

print("BẢNG KIỂM TRA GIÁ TRỊ BẤT THƯỜNG")
display(qa_checks_df)


BẢNG KIỂM TRA GIÁ TRỊ BẤT THƯỜNG


,table,check,affected_rows,note
0,orders,missing_order_id,0,Khóa đơn hàng không được thiếu
1,orders,duplicate_order_id,0,order_id phải duy nhất trong bảng orders
2,items,negative_price,0,Giá sản phẩm âm là bất thường
3,items,negative_freight,0,Phí vận chuyển âm là bất thường
4,items,duplicate_item_key,0,Cặp order_id và order_item_id phải duy nhất
5,reviews,invalid_review_score,0,Điểm đánh giá hợp lệ từ 1 đến 5
6,payments,negative_payment,0,Giá trị thanh toán âm là bất thường
7,payments,duplicate_payment_key,0,Khóa ghép thanh toán phải duy nhất


In [5]:
# T05.5 - Thống kê giá trị số và phân loại

print("THỐNG KÊ GIÁ SẢN PHẨM VÀ PHÍ VẬN CHUYỂN")
display(
    items[["price", "freight_value"]]
    .describe()
)

print("\nTHỐNG KÊ GIÁ TRỊ THANH TOÁN")
display(
    payments[["payment_value"]]
    .describe()
)

print("\nPHÂN PHỐI TRẠNG THÁI ĐƠN HÀNG")
display(
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="n_orders")
)

print("\nPHÂN PHỐI PHƯƠNG THỨC THANH TOÁN")
display(
    payments["payment_type"]
    .value_counts(dropna=False)
    .rename_axis("payment_type")
    .reset_index(name="n_records")
)


THỐNG KÊ GIÁ SẢN PHẨM VÀ PHÍ VẬN CHUYỂN


,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000



THỐNG KÊ GIÁ TRỊ THANH TOÁN


,payment_value
count,103886.000000
mean,154.100380
std,217.494064
min,0.000000
25%,56.790000
50%,100.000000
75%,171.837500
max,13664.080000



PHÂN PHỐI TRẠNG THÁI ĐƠN HÀNG


,order_status,n_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2



PHÂN PHỐI PHƯƠNG THỨC THANH TOÁN


,payment_type,n_records
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3


In [6]:
# T05.6 - Xuất kết quả kiểm tra QA

OUTPUT_DIR = ROOT / "reports" / "tables"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

qa_summary_df.to_csv(
    OUTPUT_DIR / "ngoc_lan_t05_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

qa_columns_df.to_csv(
    OUTPUT_DIR / "ngoc_lan_t05_columns.csv",
    index=False,
    encoding="utf-8-sig"
)

qa_checks_df.to_csv(
    OUTPUT_DIR / "ngoc_lan_t05_checks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("ĐÃ XUẤT 3 FILE QA:")
print("1. ngoc_lan_t05_summary.csv")
print("2. ngoc_lan_t05_columns.csv")
print("3. ngoc_lan_t05_checks.csv")


ĐÃ XUẤT 3 FILE QA:
1. ngoc_lan_t05_summary.csv
2. ngoc_lan_t05_columns.csv
3. ngoc_lan_t05_checks.csv


### 2.2.3. Kết luận nhiệm vụ T05

Đã thực hiện Data Profiling trên đầy đủ 9 file CSV Olist,
kiểm tra 52 cột dữ liệu và tổng hợp kết quả về kích thước,
missing values, duplicate rows, kiểu dữ liệu và bất thường.

Kết quả cho thấy các vấn đề đáng chú ý tập trung ở
trường thời gian của orders, thông tin sản phẩm,
bình luận đánh giá và các dòng trùng của geolocation.

Tám quy tắc kiểm tra khóa và giá trị bất thường
không phát hiện dòng vi phạm. Tuy nhiên, kết quả
này không đồng nghĩa dữ liệu đã hoàn toàn sạch.

Các kết quả QA được lưu thành bảng để đối chiếu với
Lam Linh. Việc làm sạch, tích hợp và tạo nhãn cuối
cùng thuộc quy trình xử lý dữ liệu của Chương 3.


## 2.3 Cơ sở thuật toán

**Dummy Classifier:** mốc tham chiếu để kiểm tra mô hình phức tạp có mang lại lợi ích hay không.

**Random Forest:** tập hợp nhiều cây quyết định bằng bootstrap và ngẫu nhiên hóa đặc trưng; phù hợp dữ liệu dạng bảng, có thể kiểm soát overfitting bằng min_samples_leaf/max_depth.

**MLP:** mạng nơ-ron nhiều lớp; học qua tối ưu hóa trọng số, cần mã hóa categorical và chuẩn hóa numeric, điều chỉnh architecture và regularization.

**Lưu ý học thuật:** không mặc định Deep Learning phải thắng Random Forest.

**Bổ sung thực hành 2.3:** Ngọc Lan trực tiếp `fit` Random Forest trên **Train chung** do giao thức Trúc Chi định nghĩa; xuất toàn bộ pipeline, xác suất RF trên Validation, cấu hình và metadata. Trúc Chi *không fit lại RF*; chỉ tải artifact và kiểm tra cùng `split_id`.


## 2.4 Các độ đo đánh giá

Với lớp dương = đơn trễ:
- **Precision** = TP/(TP+FP): chất lượng cảnh báo.
- **Recall** = TP/(TP+FN): tỷ lệ đơn trễ được phát hiện.
- **F1** = 2PR/(P+R): điều hòa Precision và Recall.
- **Average Precision (AP)**: trung bình có trọng số của Precision theo mức tăng Recall trên đường Precision–Recall (`average_precision_score`).
- **False Positives**: số đơn báo trễ nhưng thực tế đúng hạn.
- **Precision@K**: tỷ lệ đơn thật sự trễ trong Top-K xác suất lớn nhất.
- **Recall@K**: tỷ lệ đơn trễ được phát hiện trong Top-K, so với tổng số đơn trễ.

Không đồng nhất AP với ROC-AUC. Có thể báo cả ROC-AUC phụ, nhưng AP được ưu tiên nếu lớp dương ít.
**Bài thực hành 2.4:** từ xác suất RF trên Validation tính lại TP, FP, FN, TN, Precision, Recall, F1, AP, Precision@K và Recall@K. **Chỉ minh họa thực nghiệm với Validation**; người làm Test độc lập là Vĩnh Chi.


In [8]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

# Ví dụ NHÂN TẠO để minh họa cách tính – KHÔNG phải kết quả Olist.
y_true = np.array([0, 1, 1, 0, 0, 1])
y_prob = np.array([0.10, 0.80, 0.60, 0.30, 0.70, 0.90])
threshold = 0.50
y_pred = (y_prob >= threshold).astype(int)
print("Precision minh họa:", round(precision_score(y_true, y_pred, zero_division=0), 3))
print("Recall minh họa:", round(recall_score(y_true, y_pred, zero_division=0), 3))
print("F1 minh họa:", round(f1_score(y_true, y_pred, zero_division=0), 3))
print("Average Precision minh họa:", round(average_precision_score(y_true, y_prob), 3))
print("Confusion matrix minh họa:\n", confusion_matrix(y_true, y_pred))

Precision minh họa: 0.75
Recall minh họa: 1.0
F1 minh họa: 0.857
Average Precision minh họa: 0.917
Confusion matrix minh họa:
 [[2 1]
 [0 3]]


### 2.4.1 Gợi ý diễn giải hình minh họa
Giải thích mối đánh đổi khi thay đổi threshold: threshold thấp thường tăng Recall nhưng có thể tăng cảnh báo sai. **Kịch bản Top-K** thể hiện khả năng giám sát có giới hạn nhân lực. Ví dụ giả định ở trên chỉ để dạy thuật toán; bắt buộc xóa hoặc ghi rõ “minh họa” khi đưa vào slide báo cáo.
Phần ví dụ giả định trước đó chỉ để học công thức. Kết quả RF Validation có được từ các cell thực nghiệm tiếp theo, lưu riêng và có thể kiểm tra lại.


## THỰC HÀNH 2.3 – RANDOM FOREST (NGỌC LAN HUẤN LUYỆN)

**Đầu vào:** `data/processed/orders_model_ready.csv` từ Lam Linh; split Train/Validation/Test thống nhất và metadata từ `src/experiment.py`.

**Thời điểm dự báo:** `order_approved_at`. **Không dùng** `order_delivered_customer_date`, status cuối hoặc reviews làm feature.

**Trách nhiệm Ngọc Lan:** huấn luyện RF và chọn cấu hình bằng **AP Validation**, sau đó chọn threshold trên **Validation**, xuất `models/ngoc_lan_random_forest.joblib`, `rf_val_predictions.csv`, `rf_metadata.json`; bàn giao cho Trúc Chi. Không huấn luyện MLP/Dummy ở notebook này.

**Kiểm tra đầu tiên:** cần có `orders_model_ready.csv`; nếu Ch3 chưa bàn giao, dừng để yêu cầu Lam Linh hoàn thành. Nếu split đã được tạo, hệ thống kiểm tra SHA và `split_id` thay vì chia lại theo cách khác.

In [10]:
# T15.0 - KIEM TRA DU LIEU VA SPLIT CHUNG

from pathlib import Path
import json

HERE = Path.cwd().resolve()

ROOT_T15 = next(
    (
        p for p in [HERE, *HERE.parents]
        if (p / "docs" / "DATA_CONTRACT.md").is_file()
        and (p / "src" / "experiment.py").is_file()
    ),
    None
)

assert ROOT_T15 is not None, (
    "Khong tim thay thu muc goc cua du an."
)

PROCESSED_T15 = ROOT_T15 / "data" / "processed"

required_files = {
    "Model-ready": PROCESSED_T15 / "orders_model_ready.csv",
    "Feature whitelist": PROCESSED_T15 / "feature_whitelist.json",
    "Split manifest": PROCESSED_T15 / "split_manifest.csv",
    "Split metadata": PROCESSED_T15 / "split_metadata.json",
    "Shared experiment": ROOT_T15 / "src" / "experiment.py",
}

print("THU MUC DU AN:", ROOT_T15)
print("\nKIEM TRA CAC FILE DA BAN GIAO")

missing = []

for name, path in required_files.items():
    exists = path.is_file()
    print(f"{name}: {'FOUND' if exists else 'MISSING'}")

    if not exists:
        missing.append(name)

assert not missing, (
    f"Thieu file: {missing}. "
    "Can nhan du du lieu truoc khi huan luyen RF."
)

# Kiem tra split_id da dung trong nhom
with open(
    required_files["Split metadata"],
    encoding="utf-8"
) as f:
    split_metadata = json.load(f)

expected_split_id = "72c64fdd7b6de725a531"
actual_split_id = split_metadata.get("split_id")

print("\nSPLIT ID:", actual_split_id)

assert actual_split_id == expected_split_id, (
    "Split ID khong khop. "
    "Can doi chieu voi Truc Chi truoc khi tiep tuc."
)

print("\nPASS: Du du lieu va split chung cho T15.")


THU MUC DU AN: D:\UTH HK1 NAM 3\ĐO AN\Do_An_Khoa_hoc_du_lieu---GR02

KIEM TRA CAC FILE DA BAN GIAO
Model-ready: FOUND
Feature whitelist: FOUND
Split manifest: FOUND
Split metadata: FOUND
Shared experiment: FOUND

SPLIT ID: 72c64fdd7b6de725a531

PASS: Du du lieu va split chung cho T15.


In [11]:
from pathlib import Path
import sys, json, joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "docs" / "DATA_CONTRACT.md").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook trong repo hoặc thư mục notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"
from src.experiment import (NUMERIC_FEATURES, CATEGORY_FEATURES, SAFE_FEATURES,
                            get_shared_splits, choose_threshold_on_validation,
                            precision_recall_at_k)
splits = get_shared_splits(PROCESSED)
train, val = splits["train"], splits["val"]
split_id = splits["metadata"]["split_id"]
print("Split dùng chung:", split_id)
display(pd.DataFrame([{"split": k, "n": len(splits[k]),
                       "late_rate": splits[k]["is_late"].mean()}
                      for k in ("train", "val", "test")]))
print("Tuyệt đối không fit hoặc đánh giá RF trên Test trong notebook Ngọc Lan.")

Split dùng chung: 72c64fdd7b6de725a531


,split,n,late_rate
0,train,64416,0.068120
1,val,12551,0.042945
2,test,14469,0.042781


Tuyệt đối không fit hoặc đánh giá RF trên Test trong notebook Ngọc Lan.


### 2.3.1 Xây dựng pipeline Random Forest – Fit trên Train

Dùng cùng feature whitelist với Trúc Chi. Pipeline xử lý missing, one-hot categorical trước khi `RandomForestClassifier`.

**Tham số cần trình bày:** `n_estimators`, `min_samples_leaf`, `class_weight`, `random_state`; giải thích lý do và khả năng overfitting. Các cấu hình minh họa cần được chọn từ **Validation**.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

X_train = train[SAFE_FEATURES]
y_train = train["is_late"].astype(int)
X_val = val[SAFE_FEATURES]
y_val = val["is_late"].astype(int)

def rf_pipeline(min_leaf=5):
    prep = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), CATEGORY_FEATURES)
    ])
    return Pipeline([
        ("preprocess", prep),
        ("rf", RandomForestClassifier(
            n_estimators=200, min_samples_leaf=min_leaf,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42))
    ])

records, fitted = [], {}
for min_leaf in [2, 5, 10]:
    candidate = rf_pipeline(min_leaf)
    candidate.fit(X_train, y_train)  # TRAIN ONLY
    prob = candidate.predict_proba(X_val)[:, 1]  # VALIDATION ONLY
    score = float(average_precision_score(y_val, prob))
    records.append({"min_samples_leaf": min_leaf, "AP_validation": score})
    fitted[min_leaf] = candidate

cv_table = pd.DataFrame(records).sort_values(
    ["AP_validation", "min_samples_leaf"], ascending=[False, True])
display(cv_table)
best_leaf = int(cv_table.iloc[0]["min_samples_leaf"])
rf = fitted[best_leaf]
rf_val_prob = rf.predict_proba(X_val)[:, 1]
print("Đã chọn RF bằng AP trên Validation:", best_leaf, "split:", split_id)

,min_samples_leaf,AP_validation
2,10,0.067137
1,5,0.058271
0,2,0.055444


Đã chọn RF bằng AP trên Validation: 10 split: 72c64fdd7b6de725a531


#### Nhận xét cấu hình và lựa chọn Random Forest

Trong thực nghiệm này, nhóm xây dựng mô hình Random Forest với 200 cây quyết định (`n_estimators=200`). Tham số này được giữ cố định ở cả ba cấu hình nhằm tập trung so sánh ảnh hưởng của `min_samples_leaf` đến khả năng dự đoán nguy cơ giao hàng trễ.

Tham số `class_weight="balanced_subsample"` được sử dụng để điều chỉnh trọng số giữa các lớp theo từng mẫu bootstrap. Cách thiết lập này phù hợp để thử nghiệm trên dữ liệu có sự mất cân bằng giữa số đơn giao trễ và không giao trễ.

Nhóm sử dụng `random_state=42` để cố định yếu tố ngẫu nhiên trong quá trình huấn luyện, hỗ trợ khả năng tái lập kết quả khi sử dụng cùng dữ liệu và môi trường thực nghiệm.

Đối với tham số `min_samples_leaf`, nhóm thử nghiệm ba giá trị 2, 5 và 10. Tham số này quy định số lượng mẫu tối thiểu tại một nút lá của cây quyết định. Giá trị lớn hơn có thể giúp hạn chế việc mô hình học quá chi tiết các mẫu trong tập Train, tuy nhiên hiệu quả cần được kiểm chứng bằng dữ liệu Validation.

Kết quả thực nghiệm cho thấy:

- `min_samples_leaf=2`: AP Validation = 0,055444.
- `min_samples_leaf=5`: AP Validation = 0,058271.
- `min_samples_leaf=10`: AP Validation = 0,067137.

Trong ba cấu hình đã thử, `min_samples_leaf=10` đạt AP Validation cao nhất. Vì vậy, nhóm lựa chọn cấu hình này làm mô hình Random Forest chính để tiếp tục đánh giá các chỉ số Precision, Recall, F1-score, False Positive và Top-K.

Việc lựa chọn cấu hình chỉ dựa trên tập Validation. Tập Test không được sử dụng để lựa chọn tham số hoặc điều chỉnh mô hình, nhằm bảo đảm tính độc lập cho giai đoạn đánh giá của Vĩnh Chi.


### 2.4.1 Đánh giá Random Forest trên Validation

- Lớp dương: `is_late=1`.
- Chọn threshold bằng **F1 Validation**, nếu bằng nhau ưu tiên ít FP hơn.
- Báo **Precision, Recall, F1, AP, TP, FP, FN, TN** và so sánh Top-K.
- Kết quả ở đây chỉ là đánh giá Validation để phát triển phương pháp; **Vĩnh Chi** sẽ đánh giá chính thức trên frozen Test.

Chỉ đưa bảng giá trị thực sau khi chạy Olist. Không sao chép ví dụ nhân tạo thành kết quả nhóm.

In [13]:
from src.experiment import classification_metrics
rf_val_metrics = choose_threshold_on_validation(y_val, rf_val_prob)
rf_threshold = float(rf_val_metrics["threshold"])
display(pd.DataFrame([{"model": "Random Forest", **rf_val_metrics,
                       "split_id": split_id, "dataset": "Validation"}]))
k_values = sorted(set(min(k, len(val)) for k in [50, 100, 200] if k > 0))
rf_topk_val = pd.DataFrame([
    precision_recall_at_k(y_val.to_numpy(), rf_val_prob, k)
    for k in k_values
])
display(rf_topk_val)
print("FP = số đơn cảnh báo sai trên Validation:",
      rf_val_metrics["FP"])

,model,AP,precision,recall,f1,TP,FP,FN,TN,threshold,split_id,dataset
0,Random Forest,0.067137,0.065064,0.411874,0.112377,222,3190,317,8822,0.25,72c64fdd7b6de725a531,Validation


,K,TP_at_K,FP_at_K,Precision_at_K,Recall_at_K
0,50,9,41,0.180,0.016698
1,100,14,86,0.140,0.025974
2,200,27,173,0.135,0.050093


FP = số đơn cảnh báo sai trên Validation: 3190


#### 2.4.1.1. Nhận xét kết quả thực nghiệm Random Forest

**a. Kết quả đánh giá trên Validation**

Mô hình Random Forest được lựa chọn với `min_samples_leaf=10` và đánh giá trên 12.551 đơn hàng thuộc tập Validation. Trong đó có 539 đơn giao trễ, chiếm khoảng 4,29%.

Kết quả cho thấy mô hình đạt AP = 0,067137, Precision = 6,51%, Recall = 41,19% và F1-score = 0,112377 tại ngưỡng phân loại 0,25.

Theo ma trận nhầm lẫn, mô hình nhận diện đúng 222 đơn giao trễ (TP), nhưng có 3.190 đơn không trễ bị cảnh báo sai (FP). Ngoài ra, mô hình bỏ sót 317 đơn giao trễ (FN) và dự đoán đúng 8.822 đơn không trễ (TN).

**b. So sánh với mô hình Dummy Classifier**

Tỷ lệ đơn giao trễ trong tập Validation xấp xỉ 0,042945. Đây cũng là mức AP tham chiếu của Dummy Classifier sử dụng chiến lược dự đoán xác suất lớp theo tỷ lệ trong Train (`strategy="prior"`), khi đánh giá trên tập Validation này.

Random Forest đạt AP = 0,067137, cao hơn mức tham chiếu khoảng 0,042945. Điều này cho thấy RF đã học được một phần thông tin giúp xếp hạng nguy cơ giao trễ, thay vì chỉ đưa ra cùng một xác suất cho tất cả đơn hàng.

Tuy nhiên, kết quả so sánh chính thức với Dummy cần được đối chiếu với mô hình và bảng Validation do Trúc Chi cung cấp.

**c. Đánh giá khả năng ưu tiên đơn hàng theo Top-K**

Kết quả Top-K trên Validation cho thấy:

- Top-50: phát hiện đúng 9 đơn trễ, Precision@50 = 18%.
- Top-100: phát hiện đúng 14 đơn trễ, Precision@100 = 14%.
- Top-200: phát hiện đúng 27 đơn trễ, Precision@200 = 13,5%.

Các kết quả trên cho thấy việc xếp hạng theo xác suất dự đoán có thể hỗ trợ lựa chọn một nhóm đơn hàng cần được ưu tiên theo dõi.

**d. Hạn chế và khả năng ứng dụng thực tế**

Mặc dù Random Forest có cải thiện về AP so với mức tham chiếu Dummy, Precision tại ngưỡng 0,25 chỉ đạt khoảng 6,51%. Điều này có nghĩa là phần lớn cảnh báo giao trễ chưa chính xác.

Số lượng 3.190 cảnh báo sai có thể tạo áp lực cho nhân viên vận hành nếu doanh nghiệp kiểm tra tất cả các đơn được cảnh báo. Trong khi đó, Recall đạt 41,19% cho thấy mô hình vẫn bỏ sót nhiều đơn giao trễ thực tế.

Vì vậy, mô hình hiện tại chưa đủ cơ sở để triển khai tự động trong hoạt động giao nhận. Kết quả phù hợp hơn với mục đích nghiên cứu và thử nghiệm danh sách ưu tiên Top-K trong phạm vi đồ án.

**e. Kết luận**

Random Forest đã hoàn thành quá trình huấn luyện và lựa chọn cấu hình dựa trên Validation. Mô hình cho thấy khả năng phân biệt nguy cơ giao trễ ở mức nhất định, nhưng còn hạn chế về số lượng cảnh báo sai và khả năng phát hiện đầy đủ đơn trễ.

Các kết quả trên chỉ thuộc tập Validation. Tập Test được giữ độc lập để Vĩnh Chi thực hiện đánh giá cuối cùng theo quy trình chung của nhóm.


### 2.4.2 Bàn giao model RF cho Trúc Chi

**Bắt buộc:**
1. `models/ngoc_lan_random_forest.joblib`: pipeline RF đã fit bằng Train.
2. `data/processed/rf_val_predictions.csv`: `order_id`, `y_true`, `p_rf` của Validation.
3. `data/processed/rf_metadata.json`: fingerprint split, feature list, cấu hình, threshold và metrics trên Validation.
4. Có giải thích trong `reports/chapters/CH02_THEORY.md`; tách rõ thuật toán và kết quả thực nghiệm.
5. Link PR/commit và xác nhận Trúc Chi đã nhận. **Không đẩy CSV/model lớn lên GitHub** (bàn giao theo kênh nội bộ nhóm).

In [14]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
pred_val = pd.DataFrame({
    "order_id": val["order_id"].to_numpy(),
    "y_true": y_val.to_numpy(),
    "p_rf": rf_val_prob,
})
assert pred_val["order_id"].is_unique
pred_val.to_csv(PROCESSED / "rf_val_predictions.csv", index=False)
joblib.dump(rf, MODEL_DIR / "ngoc_lan_random_forest.joblib")

metadata = {
    "owner": "Ngọc Lan", "model": "rf", "split_id": split_id,
    "dataset_sha256": splits["metadata"]["csv_sha256"],
    "features": list(SAFE_FEATURES),
    "best_min_samples_leaf": best_leaf,
    "n_estimators": 200, "random_state": 42,
    "validation_metrics": rf_val_metrics,
    "validation_topk": rf_topk_val.to_dict(orient="records"),
    "validation_n": len(val),
    "threshold_origin": "Validation only"
}
(PROCESSED / "rf_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã lưu RF model, Validation predictions, metadata.")
print("Người nhận: Trúc Chi. Vĩnh Chi sẽ đánh giá Test độc lập.")

Đã lưu RF model, Validation predictions, metadata.
Người nhận: Trúc Chi. Vĩnh Chi sẽ đánh giá Test độc lập.


In [15]:
# T15.4 - KIEM TRA ARTIFACT RANDOM FOREST

import json
import joblib
import numpy as np
import pandas as pd

model_path = MODEL_DIR / "ngoc_lan_random_forest.joblib"
pred_path = PROCESSED / "rf_val_predictions.csv"
meta_path = PROCESSED / "rf_metadata.json"

# 1. Kiem tra 3 file ton tai
for path in [model_path, pred_path, meta_path]:
    assert path.is_file(), f"Thieu file: {path.name}"
    print("FOUND:", path.name)

# 2. Nap lai model va du lieu Validation da luu
rf_loaded = joblib.load(model_path)
pred_saved = pd.read_csv(
    pred_path, dtype={"order_id": "string"}
)

with open(meta_path, "r", encoding="utf-8") as f:
    metadata_saved = json.load(f)

# 3. Kiem tra metadata
assert metadata_saved["split_id"] == split_id
assert metadata_saved["features"] == list(SAFE_FEATURES)
assert metadata_saved["validation_n"] == len(val)

# 4. Kiem tra so dong va thu tu order_id
assert len(pred_saved) == len(val)
assert pred_saved["order_id"].is_unique

assert np.array_equal(
    pred_saved["order_id"].astype(str).to_numpy(),
    val["order_id"].astype(str).to_numpy()
)

assert np.array_equal(
    pred_saved["y_true"].to_numpy(),
    y_val.to_numpy()
)

# 5. Du doan lai bang model da nap
p_reloaded = rf_loaded.predict_proba(X_val)[:, 1]

max_diff = float(
    np.max(
        np.abs(
            p_reloaded - pred_saved["p_rf"].to_numpy()
        )
    )
)

assert np.allclose(
    p_reloaded,
    pred_saved["p_rf"].to_numpy(),
    atol=1e-12,
    rtol=1e-12
), "Xac suat du doan khong khop"

print("\nSplit ID:", metadata_saved["split_id"])
print("So dong Validation:", len(pred_saved))
print("So features:", len(metadata_saved["features"]))
print("Sai lech xac suat lon nhat:", max_diff)

print("\nPASS: Artifact RF nap lai va tai lap thanh cong.")


FOUND: ngoc_lan_random_forest.joblib
FOUND: rf_val_predictions.csv
FOUND: rf_metadata.json

Split ID: 72c64fdd7b6de725a531
So dong Validation: 12551
So features: 5
Sai lech xac suat lon nhat: 2.220446049250313e-16

PASS: Artifact RF nap lai va tai lap thanh cong.


### Checklist bổ sung – Ngọc Lan

- [x] RF chỉ fit trên Train chung; các bước encode/impute fit Train-only.
- [x] Có bảng 3 cấu hình RF, chọn bằng AP Validation.
- [x] Có Precision/Recall/F1/AP/FP/Top-K Validation.
- [ ] Đã bàn giao pipeline, `rf_val_predictions.csv`, `rf_metadata.json` với `split_id` và không dùng Test.
- [ ] Cập nhật GitHub Issue #3, GSheet mục 2.3–2.4 và T15.

**Trạng thái thực hiện:**

- Đã huấn luyện và lựa chọn Random Forest bằng tập Validation.
- Cấu hình được chọn: `n_estimators=200`, `min_samples_leaf=10`, `class_weight="balanced_subsample"`, `random_state=42`.
- AP Validation = 0,067137; Precision = 0,065064; Recall = 0,411874; F1 = 0,112377.
- Đã xuất đủ ba artifact RF và kiểm tra nạp lại thành công trên máy Ngọc Lan.
- Sai lệch xác suất lớn nhất khi nạp lại: `2.220446049250313e-16`.
- Chờ Trúc Chi xác nhận tiếp nhận artifact và kiểm tra độc lập.
- Chờ cập nhật tiến độ, liên kết Pull Request trên GitHub Issue #3 và Google Sheet.


In [16]:
# T15.5 - BO SUNG THONG TIN TAI LAP VA BAN GIAO

import sys
import json
import sklearn
import numpy as np
import pandas as pd
import joblib

# 1. Ghi nhan phien ban thu vien
library_versions = {
    "python": sys.version.split()[0],
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "joblib": joblib.__version__
}

# 2. Doi chieu ma bam du lieu voi split chung
assert (
    metadata_saved["dataset_sha256"]
    == splits["metadata"]["csv_sha256"]
), "Dataset SHA256 khong khop"

# 3. Lay cau hinh RF tu model da nap lai
rf_params = rf_loaded.named_steps["rf"].get_params()

model_parameters = {
    "n_estimators": rf_params["n_estimators"],
    "min_samples_leaf": rf_params["min_samples_leaf"],
    "max_depth": rf_params["max_depth"],
    "class_weight": rf_params["class_weight"],
    "random_state": rf_params["random_state"]
}

# 4. Bo sung metadata
# Chi ghi nhan kiem tra tai may Ngoc Lan
metadata_saved.update({
    "library_versions": library_versions,
    "model_parameters": model_parameters,
    "artifact_verification": {
        "checked_locally": True,
        "validation_n": int(len(pred_saved)),
        "max_probability_difference": float(max_diff),
        "test_used_for_tuning": False,
        "receiver_confirmation": "pending"
    }
})

# 5. Luu lai metadata, khong sua model joblib
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(
        metadata_saved,
        f,
        ensure_ascii=False,
        indent=2
    )

# 6. Kiem tra file JSON sau khi ghi
with open(meta_path, "r", encoding="utf-8") as f:
    metadata_check = json.load(f)

assert metadata_check["split_id"] == split_id
assert metadata_check["validation_n"] == 12551
assert metadata_check["features"] == list(SAFE_FEATURES)

print("PHIEN BAN THU VIEN:")
for name, version in library_versions.items():
    print(f"{name}: {version}")

print("\nCAU HINH RF:")
for name, value in model_parameters.items():
    print(f"{name}: {value}")

print("\nDataset SHA256: MATCH")
print("Artifact verification: PASS")
print("Truc Chi confirmation: PENDING")
print("\nPASS: Da cap nhat metadata ban giao.")


PHIEN BAN THU VIEN:
python: 3.12.2
scikit_learn: 1.9.1
numpy: 2.5.4
pandas: 2.3.3
joblib: 1.6.0

CAU HINH RF:
n_estimators: 200
min_samples_leaf: 10
max_depth: None
class_weight: balanced_subsample
random_state: 42

Dataset SHA256: MATCH
Artifact verification: PASS
Truc Chi confirmation: PENDING

PASS: Da cap nhat metadata ban giao.


#### Hướng dẫn nạp lại Random Forest cho Trúc Chi

Mô hình Random Forest đã được lưu dưới dạng Pipeline hoàn chỉnh, bao gồm các bước xử lý đặc trưng và bộ phân loại đã huấn luyện.

**Các tệp bàn giao:**

- `models/ngoc_lan_random_forest.joblib`
- `data/processed/rf_val_predictions.csv`
- `data/processed/rf_metadata.json`

**Cách nạp lại mô hình:**

```python
import joblib
from pathlib import Path

model_path = Path("models/ngoc_lan_random_forest.joblib")
rf_model = joblib.load(model_path)

# X_val phải sử dụng đúng 5 đặc trưng
# và đúng thứ tự dữ liệu Validation chung.
p_rf = rf_model.predict_proba(X_val)[:, 1]
```

Khi tiếp nhận, Trúc Chi cần kiểm tra:

1. Split ID: `72c64fdd7b6de725a531`.
2. SHA256 của dữ liệu phải khớp với metadata split.
3. Danh sách đầu vào gồm đúng 5 đặc trưng của nhóm.
4. Số dòng Validation: 12.551.
5. Thứ tự `order_id`, nhãn `y_true` và xác suất `p_rf` phải khớp với tệp dự đoán đã bàn giao.

Mô hình đã được kiểm tra nạp lại và tái lập xác suất trên máy người huấn luyện. Việc tiếp nhận và xác nhận độc lập của Trúc Chi hiện chưa hoàn thành.

**Lưu ý:** Không huấn luyện lại Random Forest và không sử dụng Test để lựa chọn tham số hay threshold.


## 2.5 Nghiên cứu liên quan
**Công việc:** lựa chọn **3–5 công trình** gần chủ đề giao hàng trễ, logistics/last-mile hoặc dự báo nhị phân trên dữ liệu bảng.

Bảng tổng quan bắt buộc có:
- Tác giả, năm và đường dẫn DOI/URL.
- Bộ dữ liệu và bối cảnh nghiên cứu.
- Mô hình, độ đo và đóng góp.
- Giới hạn và điểm khác so với đồ án của nhóm.

**Chưa có bài báo được nhóm xác minh** trong notebook này; không tự tạo tên tác giả hay kết quả từ nguồn chưa đọc.

### 2.5.1. Nghiên cứu của Salari và cộng sự (2022)

**Tên công trình:** Real-Time Delivery Time Forecasting and Promising in Online Retailing: When Will Your Package Arrive?

**Tác giả:** Nooshin Salari, Sheng Liu và Zuo-Jun Max Shen (2022).

**Nguồn:** https://doi.org/10.1287/msom.2022.1081

**Dữ liệu và phương pháp:** Nghiên cứu sử dụng dữ liệu thực tế từ nền tảng JD.com. Các tác giả áp dụng phương pháp dự báo phân phối thời gian giao hàng dựa trên mô hình cây và Quantile Regression Forest, kết hợp với việc lựa chọn ngày giao hàng cam kết theo chi phí.

**Kết quả:** Trong mô phỏng được công bố, chính sách đề xuất có khả năng cải thiện doanh số khoảng 6,1% so với chính sách hiện tại.

**Hạn chế:** Nghiên cứu tập trung vào dự báo thời gian giao hàng và tối ưu ngày cam kết, khác với bài toán phân loại đơn giao trễ theo nhãn `is_late` của nhóm.

**Khả năng áp dụng:** Công trình cho thấy ý nghĩa của việc dự báo sớm trong hỗ trợ quyết định vận hành. Đây là cơ sở tham khảo cho mục tiêu nhận diện đơn có nguy cơ trễ tại `order_approved_at`.

### 2.5.2. Nghiên cứu của Saini và cộng sự (2026)

**Tên công trình:** Interpretable and leakage-aware prediction of shipment delays in global supply chains: a Random Forest–SHAP workflow with multi-model benchmarking on the DataCo dataset.

**Tác giả:** Divyansh Saini, Harman Singh và Bhavya Agarwal (2026).

**Nguồn:** https://doi.org/10.1108/MSCRA-04-2026-0035

**Dữ liệu và phương pháp:** Nghiên cứu sử dụng 172.765 bản ghi vận chuyển hoàn tất từ DataCo. Các tác giả kiểm tra thời điểm xuất hiện của từng đặc trưng, huấn luyện Random Forest và so sánh với các mô hình khác trong quy trình hạn chế rò rỉ dữ liệu.

**Kết quả:** Random Forest khi loại bỏ thông tin hậu nghiệm đạt R² = 0,2719. Khi bổ sung các trường hậu nghiệm, R² tăng lên 0,7589. Sự chênh lệch cho thấy rò rỉ dữ liệu có thể khiến kết quả đánh giá cao hơn khả năng dự đoán thực tế.

**Hạn chế:** Công trình nghiên cứu độ trễ xuất hàng và sử dụng bài toán hồi quy. Vì vậy, không thể so sánh trực tiếp R² với AP hoặc F1-score trong bài toán Olist của nhóm.

**Khả năng áp dụng:** Nhóm sử dụng năm đặc trưng có thể biết tại `order_approved_at`, chỉ fit RF trên Train và lựa chọn cấu hình bằng Validation. Đây là cách áp dụng nguyên tắc kiểm soát rò rỉ dữ liệu từ nghiên cứu.

### 2.5.3. Nghiên cứu của Zhang và cộng sự (2026)

**Tên công trình:** A Deep Tabular Learning Framework for E-Commerce Delivery Delay Prediction and Cost-Sensitive Operational Risk Mitigation.

**Tác giả:** Yuxuan Zhang, Yangxin Wang và Yuanyuan Wang (2026).

**Nguồn:** https://doi.org/10.3390/a19070597

**Dữ liệu và phương pháp:** Nghiên cứu sử dụng dữ liệu logistics thương mại điện tử công khai trên Kaggle, đề xuất mô hình Gated Feature Interaction Multilayer Perceptron (GFI-MLP) để dự đoán nguy cơ giao hàng trễ trên dữ liệu bảng.

**Kết quả:** Công trình báo cáo AUC = 0,8464 và F1-score = 0,8236. Kết quả cho thấy tiềm năng của kiến trúc học sâu trong việc khai thác mối quan hệ phức tạp giữa các đặc trưng.

**Hạn chế:** Nghiên cứu sử dụng dữ liệu khác Olist và kiến trúc GFI-MLP phức tạp hơn MLP thông thường được triển khai trong nhóm. Vì vậy, không nên so sánh trực tiếp điểm số giữa hai bộ dữ liệu.

**Khả năng áp dụng:** Nghiên cứu cung cấp cơ sở tham khảo khi nhóm so sánh Random Forest và MLP trên cùng tập Train/Validation/Test. Việc huấn luyện MLP thuộc trách nhiệm của Tấn Lên, không phải Ngọc Lan.

### 2.5.4. Tổng hợp và định hướng áp dụng

Qua ba nghiên cứu, nhóm nhận thấy việc dự đoán giao hàng trễ cần quan tâm đến cả chất lượng mô hình và khả năng ứng dụng trong vận hành.

Nghiên cứu của Salari và cộng sự nhấn mạnh vai trò của dự báo sớm. Saini và cộng sự cho thấy tầm quan trọng của việc kiểm soát rò rỉ dữ liệu. Trong khi đó, Zhang và cộng sự cung cấp thêm cơ sở về khả năng áp dụng mạng nơ-ron đối với dữ liệu bảng.

Từ những cơ sở trên, đồ án sử dụng Random Forest và MLP để dự đoán nguy cơ giao hàng trễ tại thời điểm `order_approved_at`, với Dummy Classifier làm mô hình tham chiếu.

Các mô hình sử dụng cùng giao thức dữ liệu. Nhóm lựa chọn cấu hình và threshold bằng Validation, sau đó để Vĩnh Chi đánh giá độc lập trên Test.

Do dữ liệu mất cân bằng, nhóm ưu tiên Average Precision, Precision, Recall, F1-score, False Positive và Top-K thay vì chỉ dựa trên Accuracy.

**Ghi chú:** Cell `literature_template` ngay bên dưới là bảng mẫu ban đầu của nhóm trưởng. Bảng tổng hợp đã được bổ sung tại cell tiếp theo mà không thay đổi code gốc.

In [17]:
import pandas as pd
literature_template = pd.DataFrame(columns=[
    "tac_gia_nam", "tieu_de", "doi_url", "du_lieu", "mo_hinh", "do_do",
    "dong_gop", "han_che", "lien_he_de_tai"
])
display(literature_template)
print("Mẫu bảng, chưa điền tài liệu tham khảo chưa kiểm chứng.")

,tac_gia_nam,tieu_de,doi_url,du_lieu,mo_hinh,do_do,dong_gop,han_che,lien_he_de_tai


Mẫu bảng, chưa điền tài liệu tham khảo chưa kiểm chứng.


In [18]:
literature_template = pd.DataFrame(columns=[
    "tac_gia_nam", "tieu_de", "doi_url", "du_lieu",
    "mo_hinh", "do_do", "dong_gop",
    "han_che", "lien_he_de_tai"
])

In [19]:
# T25 - Hoan thien bang tong quan 3 nghien cuu


literature_template = pd.DataFrame([
    {
        "tac_gia_nam": "Salari et al. (2022)",
        "tieu_de": "Real-Time Delivery Time Forecasting and Promising in Online Retailing",
        "doi_url": "https://doi.org/10.1287/msom.2022.1081",
        "du_lieu": "JD.com",
        "mo_hinh": "Tree-based models, Quantile Regression Forest",
        "do_do": "Do chinh xac du bao; doanh so mo phong",
        "dong_gop": "Chinh sach cam ket giao hang co the tang doanh so 6,1% trong mo phong",
        "han_che": "Du bao thoi gian giao; khong phan loai is_late",
        "lien_he_de_tai": "Vai tro cua du bao som trong van hanh"
    },
    {
        "tac_gia_nam": "Saini et al. (2026)",
        "tieu_de": "Interpretable and leakage-aware prediction of shipment delays",
        "doi_url": "https://doi.org/10.1108/MSCRA-04-2026-0035",
        "du_lieu": "DataCo, 172765 ban ghi",
        "mo_hinh": "Random Forest, SHAP, multi-model benchmarking",
        "do_do": "R2, MAE, RMSE",
        "dong_gop": "R2 = 0,2719 khi khong leakage; 0,7589 khi co leakage",
        "han_che": "Bai toan hoi quy dispatch delay, khac Olist",
        "lien_he_de_tai": "Kiem soat feature theo thoi diem order_approved_at"
    },
    {
        "tac_gia_nam": "Zhang et al. (2026)",
        "tieu_de": "A Deep Tabular Learning Framework for E-Commerce Delivery Delay Prediction",
        "doi_url": "https://doi.org/10.3390/a19070597",
        "du_lieu": "Du lieu e-commerce shipping tren Kaggle",
        "mo_hinh": "GFI-MLP",
        "do_do": "AUC, F1-score",
        "dong_gop": "AUC = 0,8464; F1 = 0,8236",
        "han_che": "Khac Olist va MLP co ban cua nhom",
        "lien_he_de_tai": "Co so tham khao so sanh RF va MLP"
    }
], columns=literature_template.columns)

assert len(literature_template) == 3
assert literature_template.notna().all().all()
assert literature_template["doi_url"].str.startswith(
    "https://doi.org/"
).all()

display(literature_template)
print("PASS: Da tong hop day du 3 nghien cuu.")


,tac_gia_nam,tieu_de,doi_url,du_lieu,mo_hinh,do_do,dong_gop,han_che,lien_he_de_tai
0,Salari et al. (2022),Real-Time Delivery Time Forecasting and Promis...,https://doi.org/10.1287/msom.2022.1081,JD.com,"Tree-based models, Quantile Regression Forest",Do chinh xac du bao; doanh so mo phong,Chinh sach cam ket giao hang co the tang doanh...,Du bao thoi gian giao; khong phan loai is_late,Vai tro cua du bao som trong van hanh
1,Saini et al. (2026),Interpretable and leakage-aware prediction of ...,https://doi.org/10.1108/MSCRA-04-2026-0035,"DataCo, 172765 ban ghi","Random Forest, SHAP, multi-model benchmarking","R2, MAE, RMSE","R2 = 0,2719 khi khong leakage; 0,7589 khi co l...","Bai toan hoi quy dispatch delay, khac Olist",Kiem soat feature theo thoi diem order_approve...
2,Zhang et al. (2026),A Deep Tabular Learning Framework for E-Commer...,https://doi.org/10.3390/a19070597,Du lieu e-commerce shipping tren Kaggle,GFI-MLP,"AUC, F1-score","AUC = 0,8464; F1 = 0,8236",Khac Olist va MLP co ban cua nhom,Co so tham khao so sanh RF va MLP


PASS: Da tong hop day du 3 nghien cuu.


## PHỤ LỤC A – CHECKLIST NGHIỆM THU NỘI BỘ CHƯƠNG 2

**Mục đích:** Kiểm tra tiến độ thực hiện Chương 2 và nhiệm vụ T15 – Random Forest. Đây là checklist quản lý nội bộ, không thuộc phần lý thuyết chính của báo cáo.

### A.1. Kiểm tra nội dung Chương 2

- [x] Có định nghĩa và ký hiệu được giải thích bằng lời, kèm nguồn tham khảo.
- [ ] Đối chiếu lần cuối nội dung thuật toán với giao thức và code tích hợp Chương 4 của Trúc Chi.
- [ ] Đối chiếu định nghĩa các độ đo với báo cáo Chương 5 của Vĩnh Chi.
- [x] Có 3 nghiên cứu học thuật liên quan, kèm DOI/URL và bảng tổng hợp.
- [x] Có liên hệ các phương pháp nghiên cứu với bài toán dự đoán giao hàng trễ trên Olist.
- [ ] Hoàn thiện `reports/chapters/CH02_THEORY.md` và cập nhật liên kết Pull Request.

### A.2. Kiểm tra thực nghiệm Random Forest – T15

- [x] Đã nhận dữ liệu Model-Ready và split chung.
- [x] Random Forest chỉ được huấn luyện trên Train.
- [x] Đã thử 3 cấu hình Random Forest và chọn theo AP Validation.
- [x] Đã đánh giá Precision, Recall, F1, AP, FP và Top-K trên Validation.
- [x] Đã lựa chọn threshold bằng Validation.
- [x] Đã xuất đủ 3 artifact Random Forest.
- [x] Đã kiểm tra nạp lại mô hình và tái lập xác suất trên máy Ngọc Lan.
- [ ] Trúc Chi xác nhận nhận đủ artifact và đối chiếu thành công.
- [ ] Cập nhật GitHub Issue #3 và trạng thái nhiệm vụ T15 trên Google Sheet.
- [ ] Hoàn thành PR sạch và được nhóm trưởng review.

### A.3. Kết quả thực nghiệm đã xác nhận

- Split ID: `72c64fdd7b6de725a531`.
- Số đơn Validation: 12.551.
- Cấu hình được chọn: `n_estimators=200`, `min_samples_leaf=10`, `class_weight="balanced_subsample"`, `random_state=42`.
- AP Validation: 0,067137.
- Precision: 0,065064.
- Recall: 0,411874.
- F1-score: 0,112377.
- Threshold: 0,25.
- False Positive: 3.190.
- Sai lệch xác suất lớn nhất khi nạp lại model: `2.220446049250313e-16`.

### A.4. Trạng thái bàn giao

**Đã thực hiện:** Huấn luyện, đánh giá Validation, xuất artifact và kiểm tra tái lập cục bộ.

**Chưa hoàn thành:** Xác nhận tiếp nhận của Trúc Chi, hoàn thiện báo cáo Markdown, cập nhật Issue #3 và nghiệm thu Pull Request.

**Nguyên tắc:** Không sử dụng Test để điều chỉnh Random Forest. Việc đánh giá trên Test thuộc trách nhiệm của Vĩnh Chi.

**Ghi chú:** Checklist này chỉ dùng để theo dõi tiến độ nội bộ; không thay thế nội dung báo cáo khoa học Chương 2.
